# GoalPath-Bond: data collection

Runs the pipeline in `goalpath_data/` and returns the three model tables:

| Table | Key | Contents |
|---|---|---|
| `bond_features` | bond | terms, AGG market/par value, renormalised weight `agg_weight` (sums to 1) |
| `prices` | bond x month-end | clean/accrued/dirty price, `spread_bps`, `yield_pct`, `transaction_cost_frac`, `is_stale` |
| `yield_curve` | date | zero yields at the 8 key rates |

Settings (dates, tables, thresholds, assumptions) are all in `goalpath_data/config.py`. Needs WRDS access (`pip install wrds`).
Set `use_cache=True` to reload the last WRDS pull from `data/raw`, and `download=False` to reuse the saved AGG file.

## TODO: decide with the group

**Sample and coverage**
- [ ] **Sample window.** Prices run Jan 2025 to Dec 2025 (`END_DATE` in `config.py`) because CRSP Treasury quotes on WRDS stop at 2025-12-31. The bond universe, though, is the AGG holdings of Nov 2026. Are we fine using today's holdings with last year's prices, or should we use holdings from around Dec 2025?
- [ ] **BondRet bonds have no Dec 2025 row.** BondRet ends Nov 2025, so about 6,600 corporates stop one month early. Options: carry the last price forward and mark it stale, fill December from TRACE trades, or end the sample at Nov 2025.
- [ ] **MBS pools are only priced Jan–Feb 2025.** Enhanced TRACE MBS data (`trace.trace_spds_mbs_enhanced`) ends 2025-03-17, and the newer standard table has no CUSIPs. Options: drop MBS from the model, carry Feb prices forward, or find another source for MBS prices.
- [ ] **28 TBA positions** (CUSIPs starting `01F` or `21H`) have no prices. Exclude them, or map each one to a representative pool?
- [ ] **`MAX_STALE_MONTHS` is `None`**, so stale prices are carried forward with no limit. Choose a cap.

**Assumptions to verify (marked VERIFY / PLACEHOLDER / ASSUMED in `config.py`)**
- [ ] `HALF_SPREAD_BPS`: placeholder transaction costs, used wherever TRACE has no observed bid/ask.
- [ ] `MIN_DENOM`: minimum trade sizes are assumed.
- [ ] `BONDRET_SPREAD_TO_BPS`: we assume BondRet `t_spread` is in percentage points. Check this against the WRDS docs.
- [ ] `CRSP_EXCLUDE_ITYPES = (11, 12)`: confirm these are the TIPS issue types.
- [ ] `SECTOR_RULES`: check the iShares sector → model sector mapping against the table printed in step 1.
- [ ] `MBS_CPR = 0`: pass-through cash flows assume no prepayment, which overstates MBS duration.
- [ ] The Treasury auctions feed (`features.fetch_treasury_auctions`): verify the TreasuryDirect field names.

**Known modelling gaps**
- [ ] Callable corporates are valued as bullet bonds, so their solved yields can differ from the iShares yield-to-worst. This shows up as the −6 bps mean gap for financials in `checks()`. Do we need call schedules?


In [ ]:
from data_files.pipeline import run, checks

tables = run(use_cache=False, download=True)

In [ ]:
checks(tables)

In [ ]:
tables["bond_features"].head()

In [ ]:
tables["prices"].head()

In [ ]:
tables["yield_curve"].tail()

Also in `tables`: `key_rate_changes_bps` (monthly curve changes for the covariance), `curve_params`, `dropped_bonds`, `trace_log`, `universe`, `auctions`. Everything is saved to `data/processed/*.csv`.